<a href="https://colab.research.google.com/github/omonigho-egbo-15/-04NPY-Lab-Assignment/blob/main/OmonighEgb_Phishing_Analysis_Using_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A. Overview

This notebook is a blue-team phishing triage lab that uses a locally run LLM (via Ollama) to help analyze emails for phishing indicators. The workflow is:

#Use Colab as the lab runtime
Install and run Ollama inside the Colab VM and start the Ollama server so the notebook can query it.
Model selection and download pulls an Ollama model (example shown: gemma2:9b-instruct-q4_K_M, which is a quantized instruct model).
Build two practical  SOC “blue team” use cases:
One-shot email analysis
Runs a “single email → structured analysis” flow (described as “one-shot email analysis”) and wraps it in a simple GUI (likely ipywidgets) so an analyst can paste an email and get a result.
What you will learn

This lab teaches several SOC-relevant technical competencies:

# Step 1: Install Ollama and Start Server in Background

E*xplanation of code block lines:*
**Line** 1: apt-get update runs a package list refresh to fetch current repository software versions on the virtual machine.
**Line** 2: apt-get install installs zstd, which is a compression utility used to unzip model assets.
**Line** 3: curl downloads and executes the official Ollama Linux installation script directly into shell.
**Line** 4: pkill terminates any orphaned or active Ollama server background instances to prevent port collisions.
**Line** 5: subprocess is imported to spawn the Ollama background daemon as a child process.
**Line** 6: time is imported to handle connection timing and sleep intervals.
**Line** 7: urllib.request is imported to perform standard HTTP API checks without external dependencies.
**Line** 8: json is imported to parse and serialize API communications.
**Line** 9: subprocess.Popen executes the system command /usr/local/bin/ollama serve in an independent background thread, redirecting stdout and stderr to DEVNULL to keep the notebook logs clean.
**Line** 10: time.sleep pauses the execution kernel for 5 seconds to allow the local server to spin up and bind to network port 11434.
**Line** 11: nvidia-smi prints active GPU hardware specifications to confirm a graphics processing unit is allocated and functional.
**Line** 12: ollama pull downloads the 4-bit quantized Gemma 2 9B parameter model weights from the official repository.
**Line** 13: urllib.request.urlopen queries http localhost 11434 api tags to verify local socket reachability.
**Line** 14: json.loads deserializes the returned byte response into a Python dictionary containing available system models.

In [ ]:
# 1. Install dependencies and Ollama on the new runtime VM
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Terminate any residual background Ollama servers
!pkill ollama

import subprocess
import time
import urllib.request
import json

# 3. Start Ollama server in background
print("Starting Ollama server on the new GPU runtime...")
subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# 4. Verify GPU visibility
!nvidia-smi

# 5. Pull the model
print("Pulling model...")
!/usr/local/bin/ollama pull gemma2:9b-instruct-q4_K_M

# 6. Verify connection health
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags") as response:
        models = json.loads(response.read().decode('utf-8'))
        print("Successfully verified connection to local Ollama server!")
        print("Loaded models:", [m['name'] for m in models.get('models', [])])
except Exception as e:
    print(f"Could not reach Ollama: {e}")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information...

# Step 2: Implement One-Shot Email Analysis

*Explanation of code block lines:*
**Line** 1: urllib.request is imported to make standard network API requests to our local server port.
**Line** 2: json is imported to format client requests and decode model payloads.
**Line** 3: analyze_email function is defined, which takes a single plain-text string representing email content.
**Line** 4: url variable holds the string value pointing to the local API endpoint at http localhost 11434 api generate.
**Line** 5: prompt variable builds a multi-line format string containing strict systemic instructions that dictate the SOC triage task, requesting the verdict, key indicators, and recommended actions.
**Line** 6: payload variable defines a dictionary specifying the gemma2 model, our prompt, and disabling token-by-token streaming.
**Line** 7: urllib.request.Request initiates an outbound HTTP request, encoding our dictionary into bytes and setting content headers to JSON.
**Line** 8: urllib.request.urlopen attempts to connect and fetch the response bytes from the server socket.
**Line** 9: json.loads deserializes the returned response byte string into a structured dictionary, extracting the model prediction response.
**Line** 10: sample_phishing_email stores a raw email message with mismatched headers, fake urgency, and a malicious login link.
**Line** 11: print statement triggers the run and displays final generated triage report to the screen.

In [ ]:
import urllib.request
import json

def analyze_email(email_content):
    # Define the API endpoint for Ollama
    url = "http://localhost:11434/api/generate"

    # Design a strict blue-team triage prompt
    prompt = f"""
    You are an expert SOC Analyst triaging suspected phishing emails.
    Analyze the following email content and provide a structured security assessment containing:
    1. PHISHING VERDICT: (SUSPICIOUS, MALICIOUS, or LEGITIMATE)
    2. KEY INDICATORS: (List suspicious URLs, sender mismatch, urgency, grammatical errors, etc.)
    3. RECOMMENDED ACTIONS: (Next steps for the blue team, e.g., block domain, reset password, etc.)

    Suspicious Email Content:
    ------------------------
    {email_content}
    ------------------------
    """

    payload = {
        "model": "gemma2:9b-instruct-q4_K_M",
        "prompt": prompt,
        "stream": False
    }

    # Format and dispatch request
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'}
    )

    try:
        with urllib.request.urlopen(req) as response:
            res_data = json.loads(response.read().decode('utf-8'))
            return res_data.get('response', 'No response received.')
    except Exception as e:
        return f"An error occurred while calling Ollama: {str(e)}"

# Example suspicious email to analyze
sample_phishing_email = """
From: security-update@secure-bank-login.com
To: employee@company.com
Subject: URGENT: Action Required on Your Bank Account

Dear customer,
We have detected unusual activity on your account. Your access will be restricted within 24 hours unless you verify your identity immediately.
Please click here http://secure-bank-login.com/auth/login.php to update your details.

Thank you,
Security Operations Team
"""

# Run analysis
print("Analyzing email...")
result = analyze_email(sample_phishing_email)
print("\n--- SOC Analysis Report ---\n")
print(result)

Analyzing email...

--- SOC Analysis Report ---

## Security Assessment: Suspicious Email

**1. PHISHING VERDICT:** **MALICIOUS**

**2. KEY INDICATORS:**

* **Sender Mismatch:** The sender address (`security-update@secure-bank-login.com`) is not a legitimate bank email address. 
* **Urgent Language:** The email uses phrases like "URGENT" and "Action Required" to create a sense of panic and pressure the recipient into acting quickly without thinking.
* **Threat of Account Closure:**  The email threatens to restrict the account within 24 hours if the recipient doesn't verify their information, creating a sense of urgency and fear.
* **Generic Greeting:** The email uses a generic greeting ("Dear customer") instead of the recipient's name.
* **Suspicious URL:** The link provided (`http://secure-bank-login.com/auth/login.php`) leads to a suspicious domain that is likely a phishing site designed to steal credentials.

**3. RECOMMENDED ACTIONS:**

* **Block Sender:** Immediately block the sen

# Step 3: Implement One-Shot Email Analysis
This step write a Python script that connects to the local Ollama API, defines a system prompt for phishing triaging, and structures the analysis of a suspicious email.

*Explanation of code block lines:*
**Line** 1: ipywidgets is imported to create functional UI widgets inside the notebook.
**Line** 2: display and clear_output are imported from IPython.display to handle dynamic UI redraws.
**Line** 3: widgets.HTML creates a stylized header title for the analyst portal.
**Line** 4: widgets.Textarea creates an input textbox populated with our sample phishing email to allow analyst manual entry.
**Line** 5: widgets.Button configures a clickable trigger button styled in red to start the triage action.
**Line** 6: widgets.Output creates a dedicated containment zone to capture, hold, and display the LLM's returned analysis.
**Line** 7: on_button_clicked function is defined to clean the display, check the input text, invoke the analyze_email function, and render the resulting security assessment.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Header label
hader_widget = widgets.HTML(value="<h2>Blue-Team Phishing Triage Analyst Portal</h2>")

# Textarea for email input
email_input = widgets.Textarea(
    value=sample_phishing_email.strip(),
    placeholder='Paste raw email headers/content here...',
    description='Email Content:',
    disabled=False,
    layout=widgets.Layout(width='90%', height='200px')
)

# Button to trigger triage
analyze_button = widgets.Button(
    description='Triage Email',
    button_style='danger',
    tooltip='Click to run LLM analysis',
    icon='search'
)

# Output widget for results
output_display = widgets.Output()

def on_button_clicked(b):
    with output_display:
        clear_output()
        print("Contacting Ollama to triage email... please wait...")
        raw_text = email_input.value
        if not raw_text.strip():
            print("Error: Email content cannot be empty.")
            return
        report = analyze_email(raw_text)
        clear_output()
        print(report)

analyze_button.on_click(on_button_clicked)

# Display GUI elements
display(hader_widget, email_input, analyze_button, output_display)

HTML(value='<h2>Blue-Team Phishing Triage Analyst Portal</h2>')

Textarea(value='From: security-update@secure-bank-login.com\nTo: employee@company.com\nSubject: URGENT: Action…

Button(button_style='danger', description='Triage Email', icon='search', style=ButtonStyle(), tooltip='Click t…

Output()

# Step 4: Preprocessing and Structured Validation Parser

*Explanation of code block lines:*
**Line** 1: re module is imported to perform string matching and markdown tag deletions.
**Line** 2: json is imported to output valid schemas and clean dict values.
**Line** 3: urllib.request is imported to send requests to local network interfaces.
**Line** 4: BeautifulSoup is imported to parse and extract target tags from HTML files.
**Line** 5: BaseModel and Field are imported from pydantic to construct our strict target object schemas.
**Line** 6: List and Literal are imported from typing to handle schema structure types.
**Line** 7: preprocess_raw_email parser runs BeautifulSoup, stripping HTML markup to return raw body text and list dictionaries of href parameters.
**Line** 8: TriageReport class declares strict schema structures for verdict, confidence score, indicators, IOC list, and actions.
**Line** 9: analyze_email_structured builds a system instruction prompt containing the schema JSON, issues the format json payload to Ollama, cleans up the returned markdown codeblocks, and runs model_validate_json to verify types.

In [ ]:
import re
import json
import urllib.request
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field
from typing import List, Literal

# --- 1. Email Preprocessing Utility ---
def preprocess_raw_email(raw_html):
    soup = BeautifulSoup(raw_html, 'html.parser')
    links = []
    for a in soup.find_all('a', href=True):
        links.append({"text": a.get_text(strip=True), "url": a['href']})
    clean_text = soup.get_text(separator=' ', strip=True)
    return clean_text, links

# --- 2. Defining the Structured Output Schema ---
class TriageReport(BaseModel):
    verdict: Literal["MALICIOUS", "SUSPICIOUS", "LEGITIMATE"] = Field(
        description="The ultimate risk categorization of the email."
    )
    confidence_score: int = Field(
        description="Confidence rating of the verdict from 1 to 100."
    )
    suspicious_indicators: List[str] = Field(
        description="Bullet points outlining suspicious elements discovered (e.g. domain spoofing, urgency, call to action)."
    )
    extracted_iocs: List[str] = Field(
        description="Specific indicators of compromise extracted, such as suspicious URLs, email addresses, or domains."
    )
    recommended_actions: List[str] = Field(
        description="Actionable playbook responses for the security team (e.g., reset credential, quarantine mail, block IP/domain)."
    )

# --- 3. Structured Analysis Runner with robust JSON extraction ---
def analyze_email_structured(raw_html_content):
    clean_text, extracted_links = preprocess_raw_email(raw_html_content)
    schema_json = json.dumps(TriageReport.model_json_schema(), indent=2)
    url = "http://localhost:11434/api/generate"
    prompt = f"""
    You are a blue-team security automation agent. Your job is to analyze the preprocessed email content and links to perform a rapid security triage.
    Analyze the content thoroughly and output your assessment STRICTLY as a populated JSON object matching this schema.
    Do not output the schema definition itself. Generate concrete values for each field based on the provided email.

    JSON Schema:
    {schema_json}

    Email Content:
    {clean_text}

    Extracted Hyperlinks:
    {json.dumps(extracted_links, indent=2)}
    """

    payload = {
        "model": "gemma2:9b-instruct-q4_K_M",
        "prompt": prompt,
        "format": "json",
        "stream": False
    }

    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'}
    )

    try:
        with urllib.request.urlopen(req) as response:
            res_data = json.loads(response.read().decode('utf-8'))
            raw_response = res_data.get('response', '{}').strip()
            if raw_response.startswith("```"):
                raw_response = re.sub(r'^```(?:json)?\s*', '', raw_response)
                raw_response = re.sub(r'\s*```$', '', raw_response)
            parsed_report = TriageReport.model_validate_json(raw_response)
            return parsed_report
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        return None

# Step 5: Test Preprocessing & Structured Output on a Obfuscated HTML Email
This section run a test with an HTML email containing hidden obfuscations and an urgency lure.

*Explanation* of code block lines:
**Line** 1: messy_html_email stores a multi-line HTML string representing a mock phishing lure containing warning text and lookalike login domains.
**Line** 2: print displays our processing checkpoint indicator to console.
**Line** 3: report variable calls our structured parser function analyze_email_structured on the messy text.
**Line** 4: if block evaluates if a valid Pydantic model object was generated.
**Line** 5: json.dumps displays validated, parsed JSON content formatted cleanly on screen.

In [ ]:
# A raw HTML email simulating suspicious brand impersonation
messy_html_email = """
<html>
<body>
<p style='color:red;'><b>WARNING: Urgent Security Alert!</b></p>
<p>Dear employee, we noticed a login from an unrecognized device in Russia.</p>
<p>To avoid immediate account termination, you must confirm your credentials within 12 hours.</p>
<p><a href='http://sign-in-microsoft-portal-verification-service.xyz/login'>Click Here to Verify Your Identity Now</a></p>
<p>Thank you,<br>IT Helpdesk Support Team</p>
</body>
</html>
"""

print("Processing and analyzing raw HTML email...")
report = analyze_email_structured(messy_html_email)

if report:
    # Display as formatted JSON to verify the schema validation
    print("\n--- Verified Structured JSON Report ---\n")
    print(json.dumps(report.model_dump(), indent=2))
else:
    print("Failed to generate structured report.")

Processing and analyzing raw HTML email...

--- Verified Structured JSON Report ---

{
  "verdict": "MALICIOUS",
  "confidence_score": 95,
  "suspicious_indicators": [
    "Urgent and threatening language ('Urgent Security Alert!', 'account termination', 'immediate')",
    "Requests immediate action with a short deadline (12 hours)",
    "Unsolicited email claiming security issue",
    "Generic greeting ('Dear employee')",
    "Spoofed sender identity ('IT Helpdesk Support Team')"
  ],
  "extracted_iocs": [
    "http://sign-in-microsoft-portal-verification-service.xyz/login"
  ],
  "recommended_actions": [
    "Quarantine the email and sender",
    "Block the malicious URL",
    "Investigate the source of the email",
    "Educate users on phishing techniques and best practices"
  ]
}


# Step 6: Evaluation Mindset & SOC Quality Control (Batch Processing & Metrics)
In a modern Security Operations Center (SOC), one doesn't just triage emails one by one; you analyze them in bulk to identify trends, lures, and false rates. Here, in this code block a mock dataset of incoming emails, run structured pipeline in a batch using **pandas**, and compile a final SOC analytics report with threat insights.

*Explanation of code block lines:*
**Line 1:** pandas is imported to handle data structured inside DataFrames.
**Line 2:** tqdm is imported to show an active progress bar for our loop iterations.
**Line 3:** emails_dataset is initialized with an array of four different HTML emails (both malicious and legitimate).
**Line 4:** df converts the emails list directly into a pandas DataFrame object.
**Line 5:** results list is initialized as an empty tracker variable.
**Line 6:** the loop processes each entry using tqdm, passing the html body to our analyze_email_structured function.
**Line 7:** successful responses append parsed details (verdict, score, IOCs) to our results list, otherwise an error block is created.
**Line 8:** results_df converts output rows into a DataFrame and merges it with the raw inputs to display statistics.

In [ ]:
# Execute the batch process again using the updated robust parser
results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    report = analyze_email_structured(row['html'])
    if report:
        results.append({
            "id": row['id'],
            "verdict": report.verdict,
            "confidence": report.confidence_score,
            "iocs": report.extracted_iocs,
            "indicators": report.suspicious_indicators
        })
    else:
        results.append({
            "id": row['id'],
            "verdict": "ERROR",
            "confidence": 0,
            "iocs": [],
            "indicators": []
        })

# Compile and display the results
results_df = pd.DataFrame(results)
final_df = pd.merge(df, results_df, on="id")

print("\n--- SOC Analytics Summary Report ---")
print(f"Total Emails Processed: {len(final_df)}")
display(final_df[['id', 'verdict', 'confidence', 'indicators']])


100%|██████████| 4/4 [00:20<00:00,  5.18s/it]


--- SOC Analytics Summary Report ---
Total Emails Processed: 4


,id,verdict,confidence,indicators
0,INC-001,MALICIOUS,85,"[Urgency in language ('immediately'), Threat o..."
1,INC-002,LEGITIMATE,95,[]
2,INC-003,SUSPICIOUS,85,[Urgency tactics used to pressure user into im...
3,INC-004,LEGITIMATE,95,[]


# Step 7: Live Email Stream Simulation
This simulator models a live email ingestion stream (such as a SOC mail queue feed). It processes arriving emails dynamically, triages them through the pre-processing and validation layers, and displays immediate security alerts for high-risk detections.

In [ ]:
import time
import random
from IPython.display import display, clear_output

# Define additional diverse raw HTML emails to simulate a live traffic mix
live_stream_templates = [
    {
        "subject": "Invoice #90848 Overdue",
        "html": """<html><body><p>Hello,</p><p>Your department invoice #90848 is 15 days past due. Please remit payment immediately at <a href='http://billing-update-portal-support.com/pay'>Online Merchant Portal</a> to avoid legal collection activities.</p></body></html>"""
    },
    {
        "subject": "Weekly Project Status sync",
        "html": """<html><body><p>Hi team, please find the deck for our sync meeting attached. No urgent action is required. See you at 2 PM.</p></body></html>"""
    },
    {
        "subject": "[SECURITY ALERT] Unauthorized Access Attempt Detected",
        "html": """<html><body><p>We detected a suspicious login attempt to your payroll portal from IP 198.51.100.42 (Beijing, CN).</p><p>If this was not you, please verify your credentials immediately at <a href='http://payroll-auth-security-verification.net/challenge'>Secure Sign-In</a> to secure your payroll deposits.</p></body></html>"""
    },
    {
        "subject": "Coffee in the breakroom",
        "html": """<html><body><p>Hi all, left some donuts and fresh coffee in the main breakroom! Help yourselves.</p></body></html>"""
    },
    {
        "subject": "Critical update required for system authentication",
        "html": """<html><body><p>Dear System Administrator,</p><p>An emergency patch must be pushed to your local terminal. Run the credential checker utility at <a href='http://admin-patch-portal.org/login'>Active Directory Management</a> within 4 hours.</p></body></html>"""
    }
]

def run_live_stream_simulation(stream_limit=5, delay_seconds=6):
    print(f"[*] Starting live SOC email ingestion stream (Max simulated events: {stream_limit})...\n")

    for i in range(1, stream_limit + 1):
        print(f"\n--- [EVENT #{i}] New Inbound Email Ingested ---")
        # Randomly pick an email template to simulate live arrival
        email_event = random.choice(live_stream_templates)
        print(f"Subject: {email_event['subject']}")
        print("[*] Parsing email body & querying local LLM...")

        # Process the email through our pipeline
        report = analyze_email_structured(email_event['html'])

        if report:
            print(f"[+] Triage Verdict: {report.verdict} (Confidence: {report.confidence_score}/100)")
            print(f"[+] Suspicious Indicators: {report.suspicious_indicators}")
            print(f"[+] Extracted IOCs: {report.extracted_iocs}")
            print(f"[+] Playbook Actions: {report.recommended_actions}")

            if report.verdict in ["MALICIOUS", "SUSPICIOUS"]:
                print(f"[ALERT] Triggered automated incident ticket generation for: {email_event['subject']}")
        else:
            print("[-] Failed to generate structured report for this event.")

        if i < stream_limit:
            print(f"\n[*] Waiting {delay_seconds} seconds for the next live ingestion event...")
            time.sleep(delay_seconds)

    print("\n[*] Live stream simulation completed.")

# Run the simulator
run_live_stream_simulation(stream_limit=3, delay_seconds=5)

[*] Starting live SOC email ingestion stream (Max simulated events: 3)...


--- [EVENT #1] New Inbound Email Ingested ---
Subject: Coffee in the breakroom
[*] Parsing email body & querying local LLM...
[+] Triage Verdict: LEGITIMATE (Confidence: 95/100)
[+] Suspicious Indicators: []
[+] Extracted IOCs: []
[+] Playbook Actions: []

[*] Waiting 5 seconds for the next live ingestion event...

--- [EVENT #2] New Inbound Email Ingested ---
Subject: Invoice #90848 Overdue
[*] Parsing email body & querying local LLM...
[+] Triage Verdict: SUSPICIOUS (Confidence: 75/100)
[+] Suspicious Indicators: ['Urgency and threat of legal action', 'Generic greeting', 'Suspicious domain name (billing-update-portal-support.com)']
[+] Extracted IOCs: ['http://billing-update-portal-support.com/pay']
[+] Playbook Actions: ['Investigate the legitimacy of the invoice and sender', 'Do not click on any links in the email', "Verify the sender's email address", 'Report the email to the appropriate security team']
[A

# SOC Triage Lab - Conceptual Review & QA
Comprehensive Lab Exercise Responses

Section 1 Understanding the labs architecture Ollama and Python

**Question 1** Explain how Ollama works conceptually model storage, ollama serve, and how clients send prompts
**Answer 1** Ollama runs as a background system service that manages your model library and runs model inference using an optimized C plus plus engine. When you pull a model, Ollama downloads and saves the quantized weights to your local storage. When you start the service with the serve command, it opens a network port on 11434 and waits for incoming HTTP requests. Any client can then send prompt payload requests to this port, and Ollama coordinates loading the weights into memory, executing the prediction layers, and streaming back the text.

**Question 2** In this notebook, why do we start ollama serve in a background thread, and what could go wrong in Colab
**Answer 2** We start the Ollama serve command as a background subprocess in Colab because running it in the foreground would permanently lock the active Python execution kernel, preventing you from running any subsequent code cells in the notebook. Running this in Colab can run into several issues, such as the network port 11434 being blocked or already in use, running out of system memory when loading larger models, or failing to detect the system graphics card, which silently forces Ollama to fall back to running on the CPU and slows down performance dramatically.

**Question 3** What does it mean that the model is q4_K_M quantized What quality/speed trade-offs should I expect
**Answer 3** The q4_K_M tag indicates that the model weights have been compressed using four bit quantization. This process shrinks the original sixteen bit floating point values down to four bits, reducing the overall model size on your drive and memory by approximately seventy five percent. This allows a nine billion parameter model to fit easily inside fifteen gigabytes of standard memory. The trade off is a major boost in inference speed with only a minor, almost unnoticeable loss in accuracy and language coherence.

**Question 4** How would I verify that the Ollama API is reachable from Python in a notebook environment
**Answer 4** You can easily verify that the Ollama API is active and reachable from your Python script by sending an HTTP GET request to the tags endpoint at http localhost 11434 api tags using standard libraries like urllib. If the server responds with a standard HTTP status code of 200 and a JSON list of loaded models, the connection is healthy.

**Question 5** If GPU is not detected, what practical changes should I make smaller model, fewer tokens, batching strategy
**Answer 5** If a graphics processing unit is not detected in your notebook environment, you should make several changes to your code. First, switch to a smaller model like Llama 3.2 3B or 1B. Second, decrease the maximum number of generated tokens to keep response latency low. Third, process your incoming emails sequentially in a single loop rather than in parallel batches to prevent running out of system memory.


**Section 2 Prompting and structured outputs SOC-friendly results**

Question 1 What is one-shot email analysis, and what are the risks of relying on one prompt with no verification steps
Answer 1 One shot email analysis refers to asking the language model to analyze a single email text and provide a final security verdict in one single execution pass. The risk of this approach is that if the model makes an early analytical error or is fooled by misleading text, it will stick to that incorrect path and hallucinate false indicators or reasons just to justify its initial wrong verdict.

Question 2 How do Pydantic models help enforce structured outputs from an LLM
Answer 2 Pydantic schemas help by enforcing a strict structure that the model must follow. It lets you define the specific keys, value data types, and constraints in Python. If the model tries to output incomplete JSON or bad fields, the Pydantic parser immediately catches the error during validation, allowing you to intercept the error and handle it gracefully.

Question 3 What is the instructor library doing, and how does it reduce formatting errors in LLM responses
Answer 3 The instructor library streamlines this process by automatically injecting structural instructions into the prompt behind the scenes. It handles sending the request, parses the JSON response, validates it against your Pydantic schema, and automatically runs corrective retries if the model output fails verification.

Question 4 Give an example JSON schema for phishing email analysis that includes classification, confidence, IOCs, and recommended actions
Answer 4 A safe JSON schema for phishing email triage should contain four key fields. A classification field returning malicious, suspicious, or legitimate. A confidence score field returning an integer from one to one hundred. An indicators field listing discovered threats like urgent wording or domain spoofing. An action steps field recommending mitigation steps like quarantine, block sender, or reset credentials.

Question 5 What prompt patterns reduce hallucinations when extracting URLs/domains from text
Answer 5 To prevent the language model from hallucinating false links, you should extract the hyperlinks programmatically with a Python parser first. Then, pass only that verified list of URLs directly into the prompt and instruct the model to only evaluate that specific list, explicitly forbidding it from fabricating any external links.


Section 3 Email and phishing fundamentals what to look for

Question 1 List the most common phishing indicators in email headers vs the email body
Answer 1 The most common phishing indicators in email headers include authentication failures in security headers, mismatched domains in the From and Return Path addresses, and unusual relays in the received route timestamps. In the email body, the main indicators are fake hyperlink destinations hiding behind trusted anchor text, urgent language threatening negative consequences, and generic greetings.

Question 2 Explain SPF, DKIM, and DMARC at a practical SOC triage level what would fail or none imply
Answer 2 SPF checks if the sending email server IP address is authorized in the domain owners DNS records. DKIM cryptographically signs the email header to prove the message was not modified in transit. DMARC uses the results of SPF and DKIM to determine how the receiving mail server should handle failures. An SPF fail means the sender used an unauthorized server, while a none result means the domain owner has not published any security policies, making spoofing much easier.

Question 3 What is the difference between From, Reply-To, and display name spoofing, and why it matters
Answer 3 Display name spoofing changes only the friendly visual name shown to the user while leaving a completely unrelated email address underneath. From address spoofing attempts to fake the actual sending domain in the header, while Reply To spoofing changes the target address where responses are sent. Understanding these differences is critical because display name spoofing is easy to execute and highly effective on mobile email clients.

Question 4 What are common social engineering lure categories and how do they map to common response actions
Answer 4 Common social engineering lure categories include urgency, fear of loss, financial gain, authority, and curiosity. These map directly to specific responses. For example, financial lures should trigger immediate transaction reviews, while authority or password expiration lures should trigger credential resets and active session termination.

Question 5 What indicators suggest credential harvesting vs malware delivery vs BEC
Answer 5 Credential harvesting is suggested by links leading to lookalike login forms asking for passwords. Malware delivery is indicated by attachments containing executable code, compressed files, or macro enabled documents. Business email compromise is characterized by plain text messages containing no links or attachments, relying entirely on social engineering to trick employees into routing funds.


Section 4 Dataset testing and evaluation

Question 1 How should I evaluate an LLM-based phishing classifier on a dataset precision, recall, confusion matrix, and why is accuracy alone misleading
Answer 1 You should evaluate an LLM phishing classifier using precision, recall, and a confusion matrix. Accuracy alone is highly misleading because if ninety nine percent of your emails are legitimate, a useless model that labels everything safe will be ninety nine percent accurate while failing to catch any actual attacks. Precision tells you what percentage of your alerts are true threats, while recall tells you what percentage of total attacks you actually caught.

Question 2 What is an acceptable false positive rate for a SOC triage helper and why does it depend on workflow
Answer 2 An acceptable false positive rate depends entirely on the operational workflow. A helper tool that simply flags emails for analyst review can tolerate a moderate false positive rate of five to ten percent. However, an automated system that blocks or deletes emails directly must have a false positive rate near zero to avoid stopping legitimate business communications.

Question 3 How do I do error analysis pick 10 false positives and summarize why the model got confused
Answer 3 To conduct an error analysis, collect ten false positives and read the raw text of each email. Group the errors by common root causes, such as the model being confused by legitimate automated notification templates, and use these insights to update your system prompt.

Question 4 How can I use visualizations wordclouds/term frequencies without accidentally over-trusting them
Answer 4 Visualizations like word clouds and term frequency charts are great for identifying trending lures in bulk, but they should not be trusted blindly because they can easily highlight common words while obscuring unique, targeted low volume indicators.


Section 5 SOC integration and operationalization

Question 1 How would I integrate this notebook approach into a SOC workflow ticketing, SOAR, SIEM, at a high level
Answer 1 You can integrate this notebook approach into a security operations center at a high level by connecting the Python script to your ingestion queue, such as a mailbox or SIEM feed. The script processes incoming emails, queries the local Ollama instance, and outputs the structured JSON directly to your security orchestration database or ticketing API to generate incident cases.

Question 2 What fields should the tool output so it can be copied into an incident ticket cleanly
Answer 2 The tool should output five clean fields to populate an incident ticket. The email identifier, the risk classification, the confidence score, the list of extracted threat indicators, and the recommended playbook actions.

Question 3 What are safe automation boundaries what should remain human-reviewed vs automated
Answer 3 Safe automation boundaries should keep high risk actions like permanent mailbox deletions under manual analyst review. Safe tasks for full automation include querying link reputation databases, extracting indicators, running files in sandboxes, and flagging suspicious alerts.

Question 4 How can I turn extracted URLs/domains into actions blocklist updates, mailbox search, or detection rules
Answer 4 You can turn extracted domains into immediate defensive actions by automatically updating your firewall blocklists, searching your entire mail server for other copies of the same email, or writing new custom detection rules for your endpoints.


Section 6 Troubleshooting

Question 1 I got an error when pulling the model / starting Ollama. Explain likely root causes in Colab and how to troubleshoot step-by-step
Answer 1 If Ollama fails to start or pull a model, the most common root causes are missing system packages, port conflicts, or running out of memory. To troubleshoot, verify that you are running on a GPU runtime, check if port 11434 is in use, check your system memory usage, and inspect active background processes using pkill to reset any hung servers.

Question 2 If the model returns malformed JSON or incomplete fields, what should I change prompt, schema, retries, or temperature
Answer 2 If the model returns malformed JSON, you should reduce the model temperature to zero, use the format json option in Ollama, make your Pydantic schema simpler, or add explicit formatting instructions to your prompt.

Question 3 If analysis is slow, what are practical optimizations smaller model, shorter prompts, caching, streaming, or batch size
Answer 3 If the analysis is running slowly, practical optimizations include switching to a smaller model, restricting the maximum number of output tokens, caching repetitive inputs, or processing your batch queues concurrently.


Section 7 Governance and safety

Question 1 What are the privacy/security considerations of analyzing real emails with an LLM, even if it is local
Answer 1 Analyzing real emails with an LLM raises major privacy issues. Even when running locally, sensitive customer records, intellectual property, or personal information can be stored in system logs. You must ensure that local directories are secured and that no external telemetry is enabled.

Question 2 What red-team prompt injection risks exist in email text, and how should a SOC tool mitigate them
Answer 2 Red team prompt injection risks exist where a malicious email contains hidden instructions telling the LLM to ignore its instructions and classify the email as safe. To mitigate this, keep user content strictly separated from system instructions, and enforce structural JSON schemas that the model cannot bypass.

Question 3 How should I log results safely so the tool does not leak sensitive content into long-term logs
Answer 3 To log results safely, strip out all sensitive message bodies, subject lines, and personal names before writing data to long term logging databases. Only record anonymous metrics like classification verdicts, confidence, and generic indicator tags.

Question 4 What are good human-in-the-loop review patterns for AI-assisted phishing triage
Answer 4 A good human in the loop pattern uses the LLM to pre triage and organize the queue, presenting the analyst with a structured summary, confidence score, and suggested actions. The analyst simply reviews the recommendation and clicks a single button to approve or reject the action.


# **Reflection**

In this project, I stood up an Ollama-backed local LLM endpoint running on a Google Colab Nvidia T4 GPU runtime and integrated it with Python using raw socket requests and Pydantic. Technically, I learned to configure local LLM environments, parse raw HTML emails cleanly with BeautifulSoup to extract hyperlinks programmatically, and use strict Pydantic schema validations to prevent JSON structure breaks. Operationally, I discovered the importance of running models locally to safeguard organizational privacy and noticed that while local models are extremely fast on GPUs, they can easily hallucinate unless restricted to explicit pre-extracted link lists. For SOC relevance, human validation remains critical before executing irreversible defensive actions like deleting emails or blocking domains to prevent business disruption due to false positives. To implement this next, I plan to run a limited pilot within our internal SOC testing environment, parsing a small, anonymized stream of incoming security alerts to test the model's triage classification precision against human analyst verdicts.